# 00 · Datos, cobertura y decisiones de comparabilidad

**Objetivo:** Medir cuánto de una diferencia de precio entre municipios se mantiene cuando se comparan productos, presentaciones y periodos equivalentes, y distinguir lo que los datos públicos permiten afirmar de lo que haría falta para evaluar una oportunidad de negocio.

**Pregunta:** Cuando un mismo producto parece más caro en un municipio que en otro, ¿qué parte de esa diferencia se sostiene al validar que se compara lo mismo, y qué información adicional se necesitaría para interpretarla como oportunidad?

Este notebook responde: ¿qué hay detrás de un «nombre de producto» en PROFECO, cuántas tiendas por municipio
hay para comparar, si las fechas de captura difieren entre municipios y qué precios son sospechosos? Con eso se fijan
el orden y los umbrales del embudo de comparabilidad. Cierra con la tabla de decisiones metodológicas.

Por qué importa: quien compara precios entre ciudades con un archivo público suele agrupar por nombre de producto y
sacar promedios. Si esa diferencia viene de comparar cosas distintas, cualquier decisión comercial basada en ella
parte de un error.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
raw = entorno.load_raw()
d_all = M.prep(raw)
d = d_all[~d_all.catalogo.isin(M.FUERA_PRINCIPAL)].copy()      # universo principal
manifest = entorno.write_manifest(raw)
params = {"semilla": M.SEMILLA, "bootstrap_replicas": 500, "repartos_azar": 200, "min_municipios": M.MIN_MUN,
          "min_tiendas": M.MIN_TIENDAS, "error_bajo": M.ERROR_BAJO, "error_alto": M.ERROR_ALTO,
          "fuera_principal": M.FUERA_PRINCIPAL, "fdr": 0.10, "min_relevante_pct": M.MIN_RELEVANTE}
_ = (META / "parametros.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Registros con precio: {len(d_all):,} · tiendas: {d_all.establecimiento_id.nunique():,} · "
      f"municipios: {d_all.municipio_id.nunique()} · nombres de producto: {d_all.producto.nunique()}")
cob = d_all.groupby("catalogo").agg(registros=("precio_num", "size"), tiendas=("establecimiento_id", "nunique"),
                                    municipios=("municipio_id", "nunique"), nombres=("producto", "nunique"),
                                    articulos=("sku_id", "nunique")).sort_values("registros", ascending=False)
display(cob)

Registros con precio: 657,602 · tiendas: 1,857 · municipios: 72 · nombres de producto: 756


,registros,tiendas,municipios,nombres,articulos
catalogo,,,,,
Básicos,330224,1044,71,202,1371
Medicamentos,152337,466,64,378,579
Útiles Escolares,60150,406,60,45,639
Electrodomésticos,38860,394,62,33,489
Frutas y Legumbres,33743,366,64,59,107
PACIC,23250,287,63,25,342
Mercados,12934,166,41,105,304
Pescados y Mariscos,5355,306,57,42,119
Tenis,510,27,21,1,68


## 1. Qué hay detrás de un nombre de producto

En PROFECO, el **nombre de producto** («Atún», «Refresco», «Folders») agrupa presentaciones, marcas y giros distintos.
Un **artículo** es la combinación exacta de nombre, presentación y marca.

In [2]:
k = d.groupby("producto").agg(presentaciones=("pres_key", "nunique"), marcas=("marca", "nunique"),
                               articulos=("sku_id", "nunique"), giros=("giro", "nunique"))
mezcla = pd.DataFrame({"% de nombres con más de una": (k > 1).mean() * 100, "mediana": k.median(), "máximo": k.max()})
display(mezcla.round(0))
ej = k.sort_values("articulos", ascending=False).head(8)
display(ej)
reg("pct_nombres_varias_presentaciones", (k.presentaciones > 1).mean() * 100, n=len(k), fuente="00_datos_y_cobertura",
    nota="% de nombres de producto (sin medicamentos) con más de una presentación")
reg("pct_nombres_varias_marcas", (k.marcas > 1).mean() * 100, n=len(k), fuente="00_datos_y_cobertura")

,% de nombres con más de una,mediana,máximo
presentaciones,61.0,2.0,68
marcas,59.0,2.0,47
articulos,77.0,4.0,68
giros,92.0,3.0,6


,presentaciones,marcas,articulos,giros
producto,,,,
Tenis,68,7,68,2
Refrigeradores,53,10,61,3
Cuaderno Profesional,11,47,58,4
Frijol,6,27,54,5
Pantallas,53,5,53,3
Jamón,5,21,52,5
Lavadoras,13,8,48,3
Carne Res,2,3,47,4


**Decisión: el universo principal excluye medicamentos.** PROFECO no registra el laboratorio: «Losartán 50 mg
con 30 tabletas» puede ser de fabricantes distintos en dos tiendas. Así, el paso «misma marca» del embudo no se puede
completar para medicamentos. Se reportan aparte, como sensibilidad. Alternativa descartada: incluirlos en el resultado
principal; serían más de la mitad de los nombres de producto y su diferencia validada seguiría mezclando productos.

## 2. Periodo: ¿las fechas de captura difieren entre municipios?

La quincena tiene 12 días con captura. Si un municipio se capturó en un día con promociones y otro no, la diferencia
podría ser de fecha y no de lugar.

In [3]:
dm = d.groupby("municipio_id").fecha.agg(dias="nunique", primero="min", ultimo="max")
print("Días con captura por municipio:")
display(dm.dias.value_counts().sort_index().rename("municipios").to_frame().T)
sem = M.cambio_entre_semanas(d)
display(pd.Series(sem).round(2).to_frame("misma tienda y artículo en las dos semanas"))
dia = M.indice_dia(d)
dia["efecto_pct"] = (np.exp(dia.efecto_log) - 1) * 100
display(dia[["registros", "efecto_pct"]].round(2).T)
F = "00_datos_y_cobertura"
reg("municipios_un_dia", (dm.dias == 1).sum(), fuente=F)
reg("municipios_hasta_dos_dias", (dm.dias <= 2).sum(), fuente=F)
reg("pct_igual_semanas", sem["pct_igual"], n=sem["pares"], fuente=F, nota="misma tienda y artículo con el mismo precio en las dos semanas")
reg("pct_cambio5_semanas", sem["pct_cambio_mayor_5"], n=sem["pares"], fuente=F)
reg("efecto_dia_max", dia.loc[dia.registros > 1000, "efecto_pct"].abs().max(), fuente=F,
    nota="mayor efecto del día de captura (días con más de 1,000 registros), en %")

Días con captura por municipio:


dias,1,2,3,4,5,6,7,8,9,10,11,12
municipios,4,14,5,6,4,4,1,2,3,2,26,1


,misma tienda y artículo en las dos semanas
pares,172224.00
tiendas,1337.00
pct_igual,81.08
pct_cambio_mayor_5,12.98
cambio_mediano_pct,0.00
cambio_medio_pct,-0.08


fecha,2026-08-16,2026-08-17,2026-08-18,2026-08-19,2026-08-20,2026-08-21,2026-08-24,2026-08-25,2026-08-26,2026-08-27,2026-08-28,2026-08-31
registros,2.0,44215.0,47277.00,46313.00,47143.00,46276.00,46114.00,44635.00,47184.00,48377.00,46843.00,40886.00
efecto_pct,-28.1,0.1,0.12,-0.01,0.01,0.04,0.19,0.02,-0.39,-0.02,-0.08,0.02


**Decisión: ajustar cada precio por el efecto del día de captura y medir si el ajuste cambia algo.**
Hay municipios capturados en uno o dos días, así que las fechas sí difieren entre municipios. Pero el precio casi no
se mueve dentro de la quincena: el efecto del día (misma cadena y mismo artículo en días distintos) es de décimas de
punto. El paso «mismo periodo» del embudo divide cada precio entre el efecto de su día. Alternativas descartadas:
usar solo una semana (perdería la mitad de los municipios) o descartar municipios con un solo día (perdería ciudades
sin razón, si la fecha no mueve el precio). El 16 de agosto tiene solo 2 registros y no pesa.

## 3. Tiendas por municipio

Con una sola tienda, «el precio del municipio» es el precio de esa tienda: su cadena, su zona y sus errores. **Unidad
de comparación** = mismo nombre, presentación, marca y giro.

In [4]:
sp = M._tiendas(d)
dist_t = sp.drop_duplicates(M.UNIDAD + ["municipio_id"]).tiendas_mun
display(dist_t.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_frame("tiendas por unidad y municipio").T)
umb = []
for t in (1, 2, 3, 5):
    x = sp[sp.tiendas_mun >= t].drop_duplicates(M.UNIDAD + ["municipio_id"])
    nm = x.groupby(M.UNIDAD).size()
    umb.append({"tiendas mínimas": t, "unidades con 10+ municipios": int((nm >= M.MIN_MUN).sum()),
                "nombres de producto": x[x.set_index(M.UNIDAD).index.isin(nm[nm >= M.MIN_MUN].index)].producto.nunique(),
                "% de pares unidad–municipio que quedan": len(x) / len(sp.drop_duplicates(M.UNIDAD + ["municipio_id"])) * 100})
umb = pd.DataFrame(umb)
display(umb.round(1))
reg("pct_unidad_mun_una_tienda", (dist_t == 1).mean() * 100, fuente=F, nota="% de pares unidad–municipio con una sola tienda")

,count,mean,std,min,10%,25%,50%,75%,90%,max
tiendas por unidad y municipio,109605.0,2.3,1.8,1.0,1.0,1.0,2.0,3.0,5.0,30.0


,tiendas mínimas,unidades con 10+ municipios,nombres de producto,% de pares unidad–municipio que quedan
0,1,3095,354,100.0
1,2,1671,311,53.5
2,3,1072,280,32.5
3,5,510,181,12.4


**Decisión: al menos 3 tiendas por municipio y 10 municipios por unidad.** El precio de cada tienda es la
mediana de sus observaciones y el del municipio, la mediana de sus tiendas. Con 3 tiendas, una sola no decide la
mediana; con 10 municipios, P10 y P90 dejan de ser el mínimo y el máximo. Sensibilidad con 2 y 5 tiendas en el
notebook 02. Alternativas descartadas: 1 tienda (el municipio sería una tienda) y 5 o más (deja fuera a la mitad de los
pares unidad–municipio y casi solo ciudades grandes).

## 4. Calidad: posibles errores de captura

In [5]:
q = M.marcar_errores(sp)
nac = sp.precio / sp.groupby(M.UNIDAD).precio.transform("median")
cal = pd.Series({"precios de tienda": len(q), "posible error contra su cadena": int(q.posible_error.sum()),
                 "posible error contra la mediana nacional": int(((nac < .5) | (nac > 2)).sum())})
display(cal.to_frame("conteo").T)
display(q[q.posible_error].sort_values("razon_cadena").head(6)[["producto", "pres_key", "marca", "cadena_comercial", "municipio_id",
                                                                  "precio", "razon_cadena"]])
reg("posibles_errores", int(q.posible_error.sum()), n=len(q), fuente=F, nota="precio fuera de 0.5×–2× la mediana de su cadena")
reg("posibles_errores_nacional", int(((nac < .5) | (nac > 2)).sum()), n=len(q), fuente=F)

,precios de tienda,posible error contra su cadena,posible error contra la mediana nacional
conteo,254987,300,792


,producto,pres_key,marca,cadena_comercial,municipio_id,precio,razon_cadena
47710,Chile Fresco,1.0 kg,S/M,Fresko la Comer,Morelos | Cuernavaca,5.7,0.129989
47774,Chile Fresco,1.0 kg,S/M,Bodega Aurrera,Tamaulipas | Reynosa,13.0,0.240741
47619,Chile Fresco,1.0 kg,S/M,Bodega Aurrera,Coahuila | Saltillo,13.0,0.240741
47725,Chile Fresco,1.0 kg,S/M,Bodega Aurrera,Oaxaca | Santa Cruz Xoxocotlán,13.0,0.240741
47665,Chile Fresco,1.0 kg,S/M,Bodega Aurrera,Estado de México | Tlalnepantla,13.0,0.240741
47772,Chile Fresco,1.0 kg,S/M,Bodega Aurrera,Tamaulipas | Reynosa,13.0,0.240741


**Decisión: el posible error se mide contra la mediana de la propia cadena en la misma unidad** (menos de la
mitad o más del doble), se marca y no se borra; el embudo muestra la diferencia con y sin esos precios. Contra la
mediana nacional se marcarían como «error» diferencias reales entre municipios o cadenas, que son justo lo que se mide.

## 5. Medida y prueba

- **Diferencia entre municipios de una unidad:** brecha P90/P10 de los precios municipales = cuánto más cuesta en
  el municipio caro típico que en el barato típico. No se usa máximo–mínimo, que depende de un solo municipio.
- **Diferencia de un nombre de producto:** mediana de sus unidades. **Del embudo:** mediana entre nombres de producto.
- **Universo fijo:** solo los nombres de producto que llegan al último paso, para que cada barra del embudo compare
  los mismos productos.
- **Comparación ingenua:** promedio simple de todos los registros con ese nombre en el municipio, como lo haría
  alguien sin revisar el dato.
- **IC90 (rango probable):** bootstrap de municipios, 500 réplicas, semilla 2026.
- **¿La diferencia es del municipio?** Prueba de rangos de Kruskal-Wallis: si los precios de tienda difieren entre
  municipios más que entre tiendas del mismo municipio. Con más de mil unidades, algunas saldrían «distintas» por azar;
  se controla con Benjamini-Hochberg al 10% (de cada 10 diferencias que se declaran, a lo más 1 sería falsa).
- **Referencia de azar:** la brecha que daría repartir al azar las mismas tiendas entre los municipios (200 repartos).

In [6]:
res, prod, un, pm = M.embudo(d)
display(res.round(1))
reg("productos_universo", len(prod), fuente=F, nota="nombres de producto que llegan al último paso (sin medicamentos)")
reg("unidades_finales", int(res.iloc[-1].unidades), fuente=F)

,paso,unidades,productos,brecha_mediana,brecha_promedio
0,ingenua,280,280,25.2,43.8
1,presentacion,1304,280,20.6,24.8
2,marca,1824,280,17.7,23.4
3,giro,2726,280,17.2,26.9
4,tiendas,1072,280,12.1,15.6
5,periodo,1072,280,12.2,15.6
6,extremos,1072,280,12.2,15.4


## 6. Decisiones metodológicas

In [7]:
t3 = umb.set_index("tiendas mínimas")
display(Markdown(f'''
| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Punto de partida | comparación ingenua: nombre de producto y promedio de todos sus registros en el municipio | es lo que haría alguien sin revisar el dato; sirve de contraste | empezar ya validado (no mostraría qué se corrige) |
| 2 | Orden del embudo | 1 presentación → 2 marca → 3 giro → 4 tiendas → 5 periodo → 6 errores | primero se define **qué** se compara (el artículo) y después **cómo** se mide su precio; además se mide cada paso por separado | otro orden sin justificar; aplicar todo de golpe (no diría qué pesa) |
| 3 | Presentación | tamaño y unidad normalizados con `unidades.py`; si no se puede normalizar, texto literal | no se suponen equivalencias | precio por kg o litro entre tamaños (el tamaño cambia el precio unitario) |
| 4 | Marca | misma marca; los cruces entre marcas se ven en los análisis 5 y 8 | dos marcas no son el mismo producto | comparar marcas sin ajuste |
| 5 | Giro | supermercado contra supermercado, mercado contra mercado | el canal cambia el precio | mezclar giros |
| 6 | Tiendas | 3 o más por municipio; precio de tienda = mediana de su quincena | {(dist_t == 1).mean()*100:.0f}% de los pares unidad–municipio tienen 1 tienda | 1 tienda; 5 o más (deja {t3.loc[5, "unidades con 10+ municipios"]} unidades contra {t3.loc[3, "unidades con 10+ municipios"]}) |
| 7 | Periodo | ajustar por el efecto del día de captura (misma cadena y artículo) | {(dm.dias <= 2).sum()} municipios con 1 o 2 días de captura; el efecto del día es de décimas | solo una semana; quitar municipios de un día |
| 8 | Errores de captura | fuera de 0.5×–2× la mediana de su cadena; marcados, con y sin ellos | contra la mediana nacional se borrarían diferencias reales | borrar; usar la mediana nacional |
| 9 | Universo | sin medicamentos (aparte, como sensibilidad); 10+ municipios por unidad; universo fijo de nombres | PROFECO no trae laboratorio | incluirlos en el principal |
| 10 | Medida | brecha P90/P10 entre municipios; mediana entre nombres de producto | estándar de la serie; no depende de un municipio | máximo–mínimo; promedio (lo dominan unos cuantos nombres) |
| 11 | Incertidumbre | IC90 por bootstrap de municipios (500, semilla 2026); Kruskal-Wallis + Benjamini-Hochberg 10%; referencia de azar | el municipio es la unidad de muestreo; más de mil pruebas exigen control de falsos positivos | prueba sin corrección (1 de cada 20 saldría distinta por azar) |
| 12 | Oportunidad | no se calcula arbitraje ni rentabilidad; tabla de qué faltaría y si está en los datos | no hay costos, márgenes ni volumen | estimar márgenes con supuestos |
'''))


| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Punto de partida | comparación ingenua: nombre de producto y promedio de todos sus registros en el municipio | es lo que haría alguien sin revisar el dato; sirve de contraste | empezar ya validado (no mostraría qué se corrige) |
| 2 | Orden del embudo | 1 presentación → 2 marca → 3 giro → 4 tiendas → 5 periodo → 6 errores | primero se define **qué** se compara (el artículo) y después **cómo** se mide su precio; además se mide cada paso por separado | otro orden sin justificar; aplicar todo de golpe (no diría qué pesa) |
| 3 | Presentación | tamaño y unidad normalizados con `unidades.py`; si no se puede normalizar, texto literal | no se suponen equivalencias | precio por kg o litro entre tamaños (el tamaño cambia el precio unitario) |
| 4 | Marca | misma marca; los cruces entre marcas se ven en los análisis 5 y 8 | dos marcas no son el mismo producto | comparar marcas sin ajuste |
| 5 | Giro | supermercado contra supermercado, mercado contra mercado | el canal cambia el precio | mezclar giros |
| 6 | Tiendas | 3 o más por municipio; precio de tienda = mediana de su quincena | 47% de los pares unidad–municipio tienen 1 tienda | 1 tienda; 5 o más (deja 510 unidades contra 1072) |
| 7 | Periodo | ajustar por el efecto del día de captura (misma cadena y artículo) | 18 municipios con 1 o 2 días de captura; el efecto del día es de décimas | solo una semana; quitar municipios de un día |
| 8 | Errores de captura | fuera de 0.5×–2× la mediana de su cadena; marcados, con y sin ellos | contra la mediana nacional se borrarían diferencias reales | borrar; usar la mediana nacional |
| 9 | Universo | sin medicamentos (aparte, como sensibilidad); 10+ municipios por unidad; universo fijo de nombres | PROFECO no trae laboratorio | incluirlos en el principal |
| 10 | Medida | brecha P90/P10 entre municipios; mediana entre nombres de producto | estándar de la serie; no depende de un municipio | máximo–mínimo; promedio (lo dominan unos cuantos nombres) |
| 11 | Incertidumbre | IC90 por bootstrap de municipios (500, semilla 2026); Kruskal-Wallis + Benjamini-Hochberg 10%; referencia de azar | el municipio es la unidad de muestreo; más de mil pruebas exigen control de falsos positivos | prueba sin corrección (1 de cada 20 saldría distinta por azar) |
| 12 | Oportunidad | no se calcula arbitraje ni rentabilidad; tabla de qué faltaría y si está en los datos | no hay costos, márgenes ni volumen | estimar márgenes con supuestos |


**Conclusión.** 61% de los nombres de producto tiene más de una presentación y 59% más de una marca, y casi la
mitad (47%) de los pares unidad–municipio tiene una sola tienda: comparar a lo bruto mezcla todo eso. Las fechas de captura difieren
entre municipios, pero el precio casi no se mueve dentro de la quincena. El embudo valida en seis pasos, sobre un
universo fijo de nombres de producto y sin medicamentos.